# Model training

### Import data and required packages

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [14]:
df = pd.read_csv('../../data/clean/clean_public_data.csv')

In [15]:
df.head()

,epglnren,sup_riscaldata,sup_raffrescata,vol_riscaldato,vol_raffrescato,sup_disperdente,rapsv,asolsut,presenza_clim_invernale,presenza_clim_estiva,...,anno_costruzione,zona_climatica,d_uso_Residenziale,tipologia_edilizia,tipologia_costruttiva,num_servizi,efficienza_media,potenza_tot,num_simulati,piano
0,193.17,72.29,72.29,261.13,261.13,176.77,0.6769,0.0550,True,True,...,1992-2005,D,True,altro,c.a. con laterizi,3,0.600000,29.62,0,ground
1,177.51,60.00,60.00,217.08,217.08,171.90,0.7918,0.0340,True,True,...,1961-1975,D,True,blocco,legno,3,0.716667,13.00,0,underground
2,128.32,65.30,27.12,241.18,102.94,79.76,0.3307,0.0601,True,True,...,1976-1985,D,True,plurifamiliare,c.a. con laterizi,3,0.593333,50.60,0,upper
3,158.86,18.81,18.81,82.50,82.50,78.21,0.9479,0.0303,True,True,...,1946-1960,D,False,plurifamiliare,muratura portante,3,0.675000,7.44,0,ground
4,218.51,73.96,73.96,331.86,331.86,262.86,0.7921,0.0445,True,True,...,1961-1975,D,True,plurifamiliare,c.a. con laterizi,3,0.753333,50.90,0,ground


### Define X and y then preprocess

In [16]:
X = df.drop(['epglnren'], axis=1)

In [17]:
y = df['epglnren']


In [18]:
# Column transformer
numerical_features = X.select_dtypes(include=['int64', 'float64', 'bool']).columns
categorical_features = X.select_dtypes(include=['object', 'category', 'str']).columns

print(f'features excluding target variable: {len(X.columns)}')
print(f'numerical features: {len(numerical_features)}')
print(f'categorical features: {len(categorical_features)}')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", categorical_transformer, categorical_features),
        ("StandardScaler", numeric_transformer, numerical_features)
    ]
)

features excluding target variable: 21
numerical features: 16
categorical features: 5


In [19]:
X = preprocessor.fit_transform(X)

In [20]:
X.shape

(1663, 56)

In [21]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((1330, 56), (333, 56))

### Create an evaluation function to present all metrics after model training

In [22]:
def evaluate_model(true, predicted):

    mse = mean_squared_error(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    
    return mae, mse, rmse, r2

In [23]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "KNN Regressor": KNeighborsRegressor(),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "SVR": SVR(),
    "CatBoost Regressor": CatBoostRegressor(verbose=0),
    "XGBoost Regressor": XGBRegressor(verbose=0),
    "LightGBM Regressor": LGBMRegressor(verbose=0)
}
model_results = []
r2_scores = []

for i in range(len(list(models))):

    model = list(models.values())[i]
    model.fit(X_train, y_train) # train the model

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # evaluate model performance
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_results.append(list(models.keys())[i])

    print('\nModel performance on training set:')
    print(f'  MAE: {model_train_mae:.4f}')
    print(f'  MSE: {model_train_mse:.4f}')
    print(f'  RMSE: {model_train_rmse:.4f}')
    print(f'  R2: {model_train_r2:.4f}')
    print ('---' * 10)
    print('Model performance on test set:')
    print(f'  MAE: {model_test_mae:.4f}')
    print(f'  MSE: {model_test_mse:.4f}')
    print(f'  RMSE: {model_test_rmse:.4f}')
    print(f'  R2: {model_test_r2:.4f}')
    
    r2_scores.append(model_test_r2)

    print('=' * 20)
    print('\n')

Linear Regression

Model performance on training set:
  MAE: 56.9024
  MSE: 6740.2298
  RMSE: 82.0989
  R2: 0.4214
------------------------------
Model performance on test set:
  MAE: 60.3852
  MSE: 7966.7917
  RMSE: 89.2569
  R2: 0.3674


Ridge Regression

Model performance on training set:
  MAE: 57.0707
  MSE: 6762.9009
  RMSE: 82.2369
  R2: 0.4195
------------------------------
Model performance on test set:
  MAE: 60.3092
  MSE: 7942.5213
  RMSE: 89.1208
  R2: 0.3693


Lasso Regression

Model performance on training set:
  MAE: 58.9456
  MSE: 7217.5413
  RMSE: 84.9561
  R2: 0.3804
------------------------------
Model performance on test set:
  MAE: 60.7778
  MSE: 7976.1746
  RMSE: 89.3094
  R2: 0.3666


KNN Regressor

Model performance on training set:
  MAE: 48.3107
  MSE: 5322.8230
  RMSE: 72.9577
  R2: 0.5431
------------------------------
Model performance on test set:
  MAE: 67.0183
  MSE: 9840.4956
  RMSE: 99.1993
  R2: 0.2186


Decision Tree Regressor

Model performance on 

c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [19:07:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [24]:
pd.DataFrame(list(zip(model_results, r2_scores)), columns=['Model', 'R2 Score']).sort_values(by='R2 Score', ascending=False)

,Model,R2 Score
8,CatBoost Regressor,0.450321
9,XGBoost Regressor,0.411265
10,LightGBM Regressor,0.400046
1,Ridge Regression,0.369312
0,Linear Regression,0.367384
2,Lasso Regression,0.366639
5,Random Forest Regressor,0.360691
3,KNN Regressor,0.218600
6,AdaBoost Regressor,0.173896
7,SVR,0.116175
